"""
🔰 データクレンジングツール（HLE対策のためのSFT用データに対応）<br>
<br>
🔧 主な処理内容:<br>
① LaTeX構文を含むかのフラグ立て（question / think / answer 各カラムごとに）<br>
② 文字化け・特殊文字の除去（別セルで処理）<br>
③ 異常に長すぎるデータのフィルタ（任意）<br>
④ 類似度チェック（別ツールと連携）<br>
<br>
✅ 利用想定:<br>
・LLMトレーニング用の前処理ステップ<br>
・"think" を含む multi-step reasoning データに対応<br>
・LaTeX構文の入出力を扱う場合に、構文が正しくパース可能か確認したい場合に便利<br>
<br>
🧪 入出力データ形式（JSONL形式）<br>
・入力ファイル: 1行ごとに {"question": "...", "think": "...", "answer": "..."} を持つJSON Lines形式<br>
・出力先: ローカルもしくはGoogle Driveに保存可<br>
<br>
📌 注意:<br>
・JSONLの文字列内に LaTeX の数式（例: \$...\$、\\(...\\)、\[...\]など）が含まれている場合、<br>
　JSONではエスケープが必要 → コラボに直接貼る際には \\ を2つ重ねるなど調整が必要です。<br>
<br>
　例:<br>
　正しい形式: "question": "The price is \$100 and the tax is \$10."<br>
　または:      "question": "The price is \\\\$100\\\\ and the tax is \\\\$10\\\\."<br>
<br>
・LaTeX構文のパースは pylatexenc ライブラリを使用します（`LatexWalker` を利用）<br>
"""


In [ ]:
# グーグルドライブ保存用 要手動連携作業 最初にやっておいて放置できるように。
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# サンプルデータ作成（データを作って確認したいときに利用）

In [ ]:
# # テストデータ作成して試したい場合に利用
# import pandas as pd


# # 各テストデータ： [input, think, output]
# # ↓ 重複や誤字・脱字、文字化け、長文、タグ付き、制御文字などを含むテストデータです
# data = [
#     # ✔️ 正常データ：簡単な方程式の解法
#     ["Solve the equation: 2x + 5 = 13",
#      "I need to isolate x. First, subtract 5 from both sides: 2x = 8. Then divide by 2: x = 4.",
#      "x = 4"],

#     # ❌ 完全一致の重複（上と同じ）
#     ["Solve the equation: 2x + 5 = 13",
#      "I need to isolate x. First, subtract 5 from both sides: 2x = 8. Then divide by 2: x = 4.",
#      "x = 4"],

#     # ✔️ 微妙な表現違いの派生データ（重複類似チェック用）
#     ["What is the derivative of x squared?",
#      "Using the power rule: d/dx(x^2) = 2x",
#      "2x"],

#     # ✔️ 同一内容だが表現違いの例
#     ["What is the derivative of x^2?",
#      "We use the power rule. x^2 becomes 2x.",
#      "2x"],

#     # ✔️ 絵文字や記号を含む入力（多言語・感情表現含む翻訳）
#     ["Translate ☺️ this to French",
#      "Try to detect emotion and translate.",
#      "Traduire cela en français"],

#     # ✔️ 短文翻訳（空のthinkテスト）
#     ["Hi",
#      "",
#      "Hello"],

#     # ❌ 異常に長い入力（長文制限チェック用）
#     ["This is an extremely long text input " * 50,
#      "Just testing",
#      "LongOutput"],

#     # ❌ 英文法の誤り（クレンジング対象：文法誤り検出）
#     ["I is go to store.",
#      "Grammar is wrong",
#      "I go to the store."],

#     # ❌ 英語の小文字始まり、脱字あり（"subtract" の前に "from" が抜けている）
#     ["solve the equation: x + 2 = 5",
#      "subtract 2 both sides.",
#      "x = 3"],

#     # ✔️ 正規フォーマットの同一問題（大文字始まり、文法正しい）
#     ["Solve the Equation: x + 2 = 5",
#      "Subtract 2 from both sides.",
#      "x = 3"],

#     # ❌ HTMLタグ付きテキスト（タグ除去テスト）
#     ["<p>Compute: 1 + 1 = 2</p>",
#      "Ensure removal of <span>tags</span>",
#      "<div>2</div>"],

#     # ❌ 制御文字混入（バイナリ系文字や改行タブなど）
#     ["Bad\x00Text",
#      "Control\tTest\nLine",
#      "Output\x07"],

#     # ❌ 記号だけの出力（意味不明な出力の異常値検出）
#     ["What is 3 * 3?",
#      "Simple multiplication",
#      "@#$%^&*"],

#     # ❌ 空白のみのデータ（異常値として除外対象）
#     ["   ", "   ", "   "],

#     # ❌ JavaScriptタグ含むXSS的入力（セキュリティフィルタ確認）
#     ["<script>alert(1)</script>",
#      "HTML should not be here",
#      "Safe output"],
# ]

# df = pd.DataFrame(data, columns=["input", "think", "output"])

# # CSV 出力
# df.to_csv("test_input.csv", index=False)

# # 表示（Colab用）
# print("✅ test_input.csv を保存しました。中身を一部表示します：")
# display(df.head(10))  # 上位10件だけ表示



In [ ]:
# # test_input.csv を読み込む
# df = pd.read_csv('test_input.csv')

# データ読み込み

In [ ]:

import pandas as pd
import glob
import json
import re

# ▼ 1. 対象JSONLファイルを取得
jsonl_files = glob.glob("./*mergeJSONL*.jsonl")
if not jsonl_files:
    print("❌ mergeJSONLファイルが見つかりません")
    exit()

df_list = []
for file in jsonl_files:
    print(f"読み込み中: {file}")
    records = []
    with open(file, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"❌ JSONエラー at {file} line {idx}: {e}")
                print(f"問題の行: {line[:120]}...")  # 行頭の一部を表示
    if records:
        df_temp = pd.DataFrame(records)
        df_list.append(df_temp)

if not df_list:
    raise ValueError("❌ 有効なレコードが見つかりません")

# ▼ 2. DataFrame結合
df = pd.concat(df_list, ignore_index=True).fillna("")

# ▼ 3. 日本語判定関数
def is_japanese(text):
    if pd.isnull(text) or text == "":
        return False
    return bool(re.search(r'[\u3040-\u30FF\u4E00-\u9FFF]', str(text)))

# ▼ 4. language列の追加
if "question" in df.columns:
    df["language"] = df["question"].apply(lambda x: "ja" if is_japanese(x) else "en")
else:
    df["language"] = "unknown"

# ▼ 5. ログ出力
print("\n--- 各列の非空件数 ---")
for col in df.columns:
    non_empty_count = df[col].apply(lambda x: x != "" and pd.notnull(x)).sum()
    print(f"{col:>12}: {non_empty_count} 件")

print("\n--- 言語別の件数 ---")
language_counts = df["language"].value_counts()
for lang, count in language_counts.items():
    print(f"{lang.upper()}: {count} 件")

# ▼ 6. use_sft / use_grpo / use_dpo の値別件数
for flag_col in ["use_sft", "use_grpo", "use_dpo"]:
    if flag_col in df.columns:
        print(f"\n--- {flag_col} の値の件数 ---")
        print(df[flag_col].value_counts(dropna=False))
    else:
        print(f"\n⚠️ {flag_col} カラムは存在しません")

print("\n--- データプレビュー ---")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.expand_frame_repr", False)
print(df.head().to_string(index=False))


読み込み中: ./instruction_dataset_0817_renumber.jsonl

--- 各列の非空件数 ---
    question: 14193 件
       think: 14193 件
      answer: 14193 件
     data_id: 14193 件
     subject: 14193 件
question_type: 9710 件
    language: 14193 件

--- 言語別の件数 ---
EN: 14193 件

⚠️ use_sft カラムは存在しません

⚠️ use_grpo カラムは存在しません

⚠️ use_dpo カラムは存在しません

--- データプレビュー ---
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      question                                                                                          

# LaTex対応（フラグ立て）

In [ ]:
!pip install pylatexenc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=7d2c1cb78683dae41332baae28aadbb06e54057224c7cd1241e76039e30d9ce6
  Stored in directory: /root/.cache/pip/wheels/b1/7a/33/9fdd892f784ed4afda62b685ae3703adf4c91aa0f524c28f03
Successfully built pylatexenc


In [ ]:

import pandas as pd
import re
from pylatexenc.latexwalker import LatexWalker, LatexWalkerError

# LaTeX構文の厳密検出（構文パーサ使用）
def is_valid_latex(text: str) -> bool:
    if not isinstance(text, str):
        return False

    # LaTeX構文にマッチする文字列を抽出
    pattern = r'(\$.*?\$|\\\((.*?)\\\)|\\\[(.*?)\\\])'
    matches = re.findall(pattern, text)

    if not matches:
        return False

    for match in matches:
        latex_expr = match[0]
        try:
            walker = LatexWalker(latex_expr)
            _, _, _ = walker.get_latex_nodes()
        except LatexWalkerError:
            return False  # パースできなかった
    return True

# DataFrameの各カラムに対してLaTeXフラグ付け＋統計出力
def detect_and_flag_latex(df: pd.DataFrame) -> pd.DataFrame:
    for col in ['question', 'think', 'answer']:
        flag_col = f'has_latex_{col}'

        # NaN除外して全体数取得
        valid_count = df[col].notna().sum()

        # LaTeX検出処理
        df[flag_col] = df[col].apply(is_valid_latex)

        # 統計出力
        latex_count = df[flag_col].sum()
        percentage = (latex_count / valid_count) * 100 if valid_count > 0 else 0
        print(f"📊 {col} カラム")
        print(f"　🔹 総件数（NaN除く）: {valid_count}")
        print(f"　🔸 LaTeX構文含む件数: {latex_count}")
        print(f"　✅ 割合: {percentage:.2f}%\n")

    return df





In [ ]:
# latexフラグ立て処理実行
df = detect_and_flag_latex(df)

# フラグ付きデータ確認
import pandas as pd
from IPython.display import display
display(df)

📊 question カラム
　🔹 総件数（NaN除く）: 14193
　🔸 LaTeX構文含む件数: 8782
　✅ 割合: 61.88%

📊 think カラム
　🔹 総件数（NaN除く）: 14193
　🔸 LaTeX構文含む件数: 8080
　✅ 割合: 56.93%

📊 answer カラム
　🔹 総件数（NaN除く）: 14193
　🔸 LaTeX構文含む件数: 871
　✅ 割合: 6.14%



,question,think,answer,data_id,subject,question_type,language,has_latex_question,has_latex_think,has_latex_answer
0,Proteins destined to be secreted move through the secretory pathway in which of the following orders?\n(A) Smooth ER → Golgi transport vesicle → Golgi cisternae → secretory vesicle → cell surface\n(B) Rough ER → Golgi transport vesicle → Golgi cisternae → secretory vesicle → cell surface\n(C) Golgi cisternae → ER transport vesicle → smooth ER → secretory vesicle → cell surface\n(D) Golgi cisternae → ER transport vesicle → rough ER → secretory vesicle → cell surface\n(E) Rough ER → smooth ER → Golgi transport vesicle → Golgi cisternae → secretory vesicle cell surface,"The secretory pathway involves proteins synthesized in the rough endoplasmic reticulum (RER) for secretion, membrane insertion, or lysosomal targeting. After synthesis in the RER, proteins are transported via vesicles to the Golgi apparatus. In the Golgi, they undergo modification and sorting through specific compartments, including the cisternae (typically referring to the cis or medial regions). From there, they are packaged into secretory vesicles that bud from the trans-Golgi network and fuse with the plasma membrane for release outside the cell.\n\nEvaluating the options:\n- (A) Incorrect: Starts with smooth ER, which is not involved in protein synthesis or initial entry for secretory proteins; rough ER is correct.\n- (B) Correct: Follows the standard pathway: synthesis in rough ER, transport via vesicles to Golgi, processing in cisternae, packaging into secretory vesicles, and exocytosis at the cell surface.\n- (C) Incorrect: Starts with Golgi cisternae, but proteins enter the pathway at the ER, not the Golgi.\n- (D) Incorrect: Starts with Golgi cisternae, reversing the pathway direction.\n- (E) Incorrect: Includes a step from rough ER to smooth ER, which is not part of the secretory pathway; proteins remain in the rough ER or proceed directly to the Golgi.",B,GRE-BioChem-Practice_BiologyAndMedicine_0000001_sft_20250803135132_000_origin,BiologyAndMedicine,Multiple-Choice,en,False,False,False
1,"In the endocytic pathway, what is the correct order of cellular compartments that a protein follows from internalization to degradation? Answer Choices: \nA. Plasma membrane → Clathrin-coated pit → Early endosome → Late endosome → Lysosome \nB. Plasma membrane → Early endosome → Clathrin-coated pit → Late endosome → Lysosome \nC. Lysosome → Late endosome → Early endosome → Clathrin-coated pit → Plasma membrane \nD. Clathrin-coated pit → Plasma membrane → Early endosome → Late endosome → Lysosome \nE. Plasma membrane → Golgi apparatus → Endoplasmic reticulum → Lysosome","The question is about the endocytic pathway, specifically the order of cellular compartments from protein internalization to degradation. Internalization begins at the plasma membrane, where endocytosis occurs, often via clathrin-coated pits. These pits invaginate and form vesicles that transport the protein to early endosomes, which are sorting compartments. From there, the protein moves to late endosomes, which are more mature and fuse with lysosomes for degradation. Reviewing the options:\n\n- Option A follows this sequence: Plasma membrane → Clathrin-coated pit → Early endosome → Late endosome → Lysosome, which aligns with the standard endocytic pathway.\n- Option B incorrectly places the early endosome before the clathrin-coated pit, which is not accurate for initial internalization.\n- Option C reverses the pathway, starting from lysosome and ending at plasma membrane, which is incorrect for internalization to degradation.\n- Option D starts with the clathrin-coated pit, but the protein begins at the plasma membrane.\n- Option E describes the secretory pathway (e.g., ER to Golgi), not the endocytic pathway.\n\nThus, option A is correct.",A,GRE-BioChem-Practice_BiologyAndMedicine_0000001_sft_20250803135132_001_origin,BiologyAndMedicine,Multiple-Choice,en,False,False,False
2,Stabiliza

In [ ]:
# おかしいカラムがないかの確認
import pandas as pd
import numpy as np

# === 1. 全行が空 or NaN/None のカラム ===
all_empty_cols = []
for c in df.columns:
    s = df[c]
    is_empty_like = s.isna() | s.apply(lambda x: x is None) | (s.astype(str) == "")
    if is_empty_like.all():
        all_empty_cols.append(c)

print("=== 全行が空 or NaN/None のカラム ===")
if all_empty_cols:
    for col in all_empty_cols:
        print(f"- {col}")
else:
    print("該当なし")

# === 2. 部分的に空 or NaN/None のカラム（割合付き） ===
print("\n=== 部分的に空 or NaN/None のカラム（割合付き） ===")
partial_empty_cols = []
for c in df.columns:
    s = df[c]
    empty_like_count = (s.isna() | s.apply(lambda x: x is None) | (s.astype(str) == "")).sum()
    if 0 < empty_like_count < len(s):
        ratio = empty_like_count / len(s) * 100
        partial_empty_cols.append((c, empty_like_count, len(s), ratio))

if partial_empty_cols:
    for col, empty_count, total, ratio in partial_empty_cols:
        print(f"- {col}: {empty_count}/{total} ({ratio:.1f}%)")
else:
    print("該当なし")

# === 3. NaNが1つでも含まれるカラム（確認用） ===
print("\n=== NaNを含むカラム ===")
nan_cols = [c for c in df.columns if df[c].isna().any()]
if nan_cols:
    for col in nan_cols:
        print(f"- {col}")
else:
    print("該当なし")


=== 全行が空 or NaN/None のカラム ===
該当なし

=== 部分的に空 or NaN/None のカラム（割合付き） ===
- question_type: 4483/14193 (31.6%)

=== NaNを含むカラム ===
該当なし


# 文字化け・特殊文字除去

In [ ]:
# ───────────────────────────────────────────────
# 特殊文字クレンジングモジュール（LaTeX考慮 / 修正可能性に基づく処理）
#
# 【処理方針】
# - LaTeX構文を含む行は構文破壊の恐れがあるため一切処理せずスキップ
# - 各列（input, think, output）に以下の分類を行う：
#
#     [修正可能な文字]
#       - タブ文字（\t）   ：空白に変換　→コーディングのデータが入る際にデータを壊すのでおやすみ
#       - HTMLタグ         ：完全除去
#
#     [修正困難な文字]
#       - 制御文字（\x00〜\x08, \x0B〜\x1F, \x7F）※\nは許容
#       - 絵文字（emoticon/symbol/flag等）
#       → フラグを立てて記録（削除はしない）
#
# - 結果として各カラムに以下を追加：
#     ・{col}_cleaned     ：クレンジング済みテキスト
#     ・{col}_issues      ：検出された問題の種類（list形式）
#     ・has_invalid_chars ：修正困難な文字が含まれているか（全体判定）
#
# 【備考】
# この処理は除外を目的とせず、「どこに問題があるか」を識別して後続の判断に役立てるためのもの
# ───────────────────────────────────────────────

import re
import pandas as pd

# ────────────────
# パターン定義
# ────────────────
_TAB_PATTERN         = re.compile(r'\t')
_CONTROL_PATTERN     = re.compile(r'[\x00-\x08\x0B-\x1F\x7F]')  # 改行(\x0A)は除外
_HTML_PATTERN        = re.compile(r'<[^>]+>')
_EMOJI_PATTERN       = re.compile(
    "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
    "]+",
    flags=re.UNICODE
)

def clean_and_flag(text: str) -> (str, list):
    """
    特殊文字をクレンジングし、修正内容の種類を返す。
    Returns:
        cleaned_text: 修正済みテキスト
        issues: ["tab", "html", "control", "emoji"]
    """
    issues = []
    original = text
    # tab,htmlはコーディングが入ると入ってくるためコメントアウトしておく
    # if _TAB_PATTERN.search(text):
    #     text = _TAB_PATTERN.sub(' ', text)
        # issues.append('tab')
    # if _HTML_PATTERN.search(text):
    #     text = _HTML_PATTERN.sub('', text)
    #     issues.append('html')
    if _CONTROL_PATTERN.search(text):
        issues.append('control')
    if _EMOJI_PATTERN.search(text):
        issues.append('emoji')

    return text, issues

def process_special_characters(df: pd.DataFrame) -> pd.DataFrame:
    """
    LaTeXを含まない行に対して、修正・判定を実施。
    結果は各列に *_cleaned, *_issues を追加し、has_invalid_chars フラグも付加。
    さらにログ出力を行う。
    """
    df = df.copy()
    total_rows = len(df)
    fixed_rows = 0
    flagged_rows = 0

    for col in ['question', 'think', 'answer']:
        cleaned_col = []
        issue_col = []

        for idx, row in df.iterrows():
            if row.get(f'has_latex_{col}'):
                cleaned_col.append(row[col])
                issue_col.append([])
                continue

            text = str(row[col])
            cleaned, issues = clean_and_flag(text)
            cleaned_col.append(cleaned)
            issue_col.append(issues)

        df[f'{col}_cleaned'] = cleaned_col
        df[f'{col}_issues'] = issue_col

    # フラグ列
    df['has_invalid_chars'] = df.apply(
        lambda row: any(
            'control' in row[f'{col}_issues'] or 'emoji' in row[f'{col}_issues']
            for col in ['question', 'think', 'answer']
        ),
        axis=1
    )
    flagged_rows = df['has_invalid_chars'].sum()

    # 修正された（tab/html）件数
    df['was_fixed'] = df.apply(
        lambda row: any(
            'tab' in row[f'{col}_issues'] or 'html' in row[f'{col}_issues']
            for col in ['question', 'think', 'answer']
        ),
        axis=1
    )
    fixed_rows = df['was_fixed'].sum()

    # ─────── ログ出力 ───────
    print("📊 特殊文字処理ログ")
    print(f"🔹 総レコード数: {total_rows} 件")
    print(f"🔧 修正された行数（tab/html）: {fixed_rows} 件")
    print(f"❌ 修正困難な文字でフラグが立った行数（control/emoji）: {flagged_rows} 件")
    print(f"✅ 残り（問題なし）: {total_rows - fixed_rows - flagged_rows} 件")

    return df

def apply_cleaned_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    *_cleaned カラムの内容で元の question/think/answer を置き換える。
    LaTeXが含まれる行はそのまま維持。
    """
    for col in ['question', 'think', 'answer']:
        cleaned_col = f"{col}_cleaned"
        df[col] = df.apply(
            lambda row: row[cleaned_col] if not row.get(f'has_latex_{col}', False) else row[col],
            axis=1
        )
    return df


In [ ]:
#特殊文字クレンジング実行
df = process_special_characters(df)
# cleanデータを反映
df = apply_cleaned_columns(df)
# 修正困難な文字を含む行だけ表示
df[df['has_invalid_chars']].head()

# 修正された input だけ確認
df[['question', 'question_cleaned', 'question_issues']].head()


📊 特殊文字処理ログ
🔹 総レコード数: 14193 件
🔧 修正された行数（tab/html）: 0 件
❌ 修正困難な文字でフラグが立った行数（control/emoji）: 0 件
✅ 残り（問題なし）: 14193 件


,question,question_cleaned,question_issues
0,Proteins destined to be secreted move through the secretory pathway in which of the following orders?\n(A) Smooth ER → Golgi transport vesicle → Golgi cisternae → secretory vesicle → cell surface\n(B) Rough ER → Golgi transport vesicle → Golgi cisternae → secretory vesicle → cell surface\n(C) Golgi cisternae → ER transport vesicle → smooth ER → secretory vesicle → cell surface\n(D) Golgi cisternae → ER transport vesicle → rough ER → secretory vesicle → cell surface\n(E) Rough ER → smooth ER → Golgi transport vesicle → Golgi cisternae → secretory vesicle cell surface,Proteins destined to be secreted move through the secretory pathway in which of the following orders?\n(A) Smooth ER → Golgi transport vesicle → Golgi cisternae → secretory vesicle → cell surface\n(B) Rough ER → Golgi transport vesicle → Golgi cisternae → secretory vesicle → cell surface\n(C) Golgi cisternae → ER transport vesicle → smooth ER → secretory vesicle → cell surface\n(D) Golgi cisternae → ER transport vesicle → rough ER → secretory vesicle → cell surface\n(E) Rough ER → smooth ER → Golgi transport vesicle → Golgi cisternae → secretory vesicle cell surface,[]
1,"In the endocytic pathway, what is the correct order of cellular compartments that a protein follows from internalization to degradation? Answer Choices: \nA. Plasma membrane → Clathrin-coated pit → Early endosome → Late endosome → Lysosome \nB. Plasma membrane → Early endosome → Clathrin-coated pit → Late endosome → Lysosome \nC. Lysosome → Late endosome → Early endosome → Clathrin-coated pit → Plasma membrane \nD. Clathrin-coated pit → Plasma membrane → Early endosome → Late endosome → Lysosome \nE. Plasma membrane → Golgi apparatus → Endoplasmic reticulum → Lysosome","In the endocytic pathway, what is the correct order of cellular compartments that a protein follows from internalization to degradation? Answer Choices: \nA. Plasma membrane → Clathrin-coated pit → Early endosome → Late endosome → Lysosome \nB. Plasma membrane → Early endosome → Clathrin-coated pit → Late endosome → Lysosome \nC. Lysosome → Late endosome → Early endosome → Clathrin-coated pit → Plasma membrane \nD. Clathrin-coated pit → Plasma membrane → Early endosome → Late endosome → Lysosome \nE. Plasma membrane → Golgi apparatus → Endoplasmic reticulum → Lysosome",[]
2,Stabilization of the unique coiled structure of an alpha helix in a protein is primarily attributed to (A) hydrogen bonding between the peptide backbone atoms (B) disulfide bridges between cysteine side chains (C) carbohydrate moieties attached to polar amino acids (D) peptide linkages that covalently bond amino acids (E) an abundance of amino acids with electrically charged side chains,Stabilization of the unique coiled structure of an alpha helix in a protein is primarily attributed to (A) hydrogen bonding between the peptide backbone atoms (B) disulfide bridges between cysteine side chains (C) carbohydrate moieties attached to polar amino acids (D) peptide linkages that covalently bond amino acids (E) an abundance of amino acids with electrically charged side chains,[]
3,Which of the following is the primary stabilizing force for the tertiary structure of a protein? \nAnswer Choices: \nA. Hydrogen bonding \nB. Disulfide bonds \nC. Hydrophobic interactions \nD. Peptide linkages that covalently bond amino acids \nE. Ionic bonds,Which of the following is the primary stabilizing force for the tertiary structure of a protein? \nAnswer Choices: \nA. Hydrogen bonding \nB. Disulfide bonds \nC. Hydrophobic interactions \nD. Peptide linkages that covalently bond amino acids \nE. Ionic bonds,[]
4,A DNA strand with the sequence 5' CGA TTG 3' would be complementary to the sequence (A) 5' GCU AAC 3' (B) 5' GCT AAC 3' (C) 5' GTT AGC 3' (D) 5' CAA TCG 3' (E) 5' CUU TCG 3',A DNA strand with the sequence 5' CGA TTG 3' would be complementary to the sequence (A) 5' GCU AAC 3' (B) 5' GCT AAC 3' (C) 5' GTT AGC 3' (D) 5' CAA TCG 3' (E) 5

In [ ]:

# ────────────────────────────────
# 🔧 異常長・短判定のしきい値設定
# ────────────────────────────────

INPUT_MIN_LEN = 10
INPUT_MAX_LEN = 10000
THINK_MIN_LEN = 100
THINK_MAX_LEN = 10000
OUTPUT_MIN_LEN = 1
OUTPUT_MAX_LEN = 2000

# NaNを許容するか
ALLOW_THINK_NAN = True


# ────────────────────────────────
# 📦 長さ異常データの除去処理（think NaN切り替え可）
# ────────────────────────────────

def remove_abnormal_length_rows(df: pd.DataFrame,
                                 allow_think_nan: bool = ALLOW_THINK_NAN):
    """
    各カラムの長さをチェックし、しきい値外の行を除外。
    thinkはNaNを許容するかどうかを外部パラメータで制御可能。

    Returns:
        cleaned_df: 正常データ
        removed_df: 除外されたデータ（error_reason付き）
    """
    cleaned_rows = []
    removed_rows = []

    for _, row in df.iterrows():
        input_text = str(row["question"]).strip()
        output_text = str(row["answer"]).strip()
        think_text = str(row["think"]).strip() if pd.notna(row["think"]) else None

        input_len = len(input_text)
        output_len = len(output_text)
        think_len = len(think_text) if think_text is not None else None

        reasons = []

        # input長チェック
        if input_len < INPUT_MIN_LEN:
            reasons.append(f"question too short ({input_len})")
        elif input_len > INPUT_MAX_LEN:
            reasons.append(f"question too long ({input_len})")

        # thinkチェック（NaN対応）
        if think_text is None:
            if not allow_think_nan:
                reasons.append("think is NaN")
        else:
            if think_len < THINK_MIN_LEN:
                reasons.append(f"think too short ({think_len})")
            elif think_len > THINK_MAX_LEN:
                reasons.append(f"think too long ({think_len})")

        # output長チェック
        if output_len < OUTPUT_MIN_LEN:
            reasons.append(f"answer too short ({output_len})")
        elif output_len > OUTPUT_MAX_LEN:
            reasons.append(f"answer too long ({output_len})")

        if reasons:
            removed = row.copy()
            removed["error_reason"] = "; ".join(reasons)
            removed_rows.append(removed)
        else:
            cleaned_rows.append(row)

    cleaned_df = pd.DataFrame(cleaned_rows).reset_index(drop=True)
    removed_df = pd.DataFrame(removed_rows).reset_index(drop=True)

    print("📊 長さチェックログ")
    print(f"✅ 正常レコード数: {len(cleaned_df)} 件")
    print(f"❌ 除外された異常長レコード数: {len(removed_df)} 件")

    return cleaned_df, removed_df


In [ ]:

# # ────────────────────────────────
# # ✅ 実行：長さチェック
# # ────────────────────────────────

# 現在は NaN を許容
df, removed_length_df = remove_abnormal_length_rows(df, allow_think_nan=True)

# NaNをNGにしたくなったら引数だけ変更
# df, removed_length_df = remove_abnormal_length_rows(df, allow_think_nan=False)



# ───────────────────────────────────────────────
# 📁 異常長データの確認・CSV出力（Colab対応）
# ───────────────────────────────────────────────

import pandas as pd

# ▼ すでに remove_abnormal_length_rows() 関数で以下が得られている前提
# df: 正常なレコード
# removed_length_df: 異常長で除外されたレコード

# 📌 1. 除外データの件数を表示
print(f"❌ 除外対象（長さ異常）件数: {len(removed_length_df)} 件")

# 📌 2. 上位5件だけ内容確認
print("\n🔍 除外された異常長レコードのサンプル:")
display(removed_length_df.head(5))  # Colabでも表形式で表示される

# 📌 3. jsonlファイルとして出力（ファイル名: removed_length_records_{timestamp}.jsonl）
import json
import os
from datetime import datetime
from zoneinfo import ZoneInfo

# 📌 JSTの現在時刻を取得（ミリ秒まで）
timestamp = datetime.now(ZoneInfo("Asia/Tokyo")).strftime('%Y%m%d_%H%M%S%f')[:-3]

# 📌 出力ファイルパス（ファイル名末尾に日時を付与）
output_path = f"/content/removed_length_records_{timestamp}.jsonl"

# 📌 JSONLとして出力
with open(output_path, 'w', encoding='utf-8') as f:
    for _, row in removed_length_df.iterrows():
        record = row.to_dict()
        json.dump(record, f, ensure_ascii=False)
        f.write('\n')

print(f"\n💾 JSONLファイルとして保存しました: {output_path}")




📊 長さチェックログ
✅ 正常レコード数: 13745 件
❌ 除外された異常長レコード数: 448 件
❌ 除外対象（長さ異常）件数: 448 件

🔍 除外された異常長レコードのサンプル:


,question,think,answer,data_id,subject,question_type,language,error_reason
0,"A woman is recovering following a fractured neck of her left femur, and is requiring assistance of one to mobilise. Her past medical history includes diabetes, rheumatoid arthritis, congestive heart failure, and hypertension. Which of these would you recommend as a most appropriate walking aid? A) Left sided crutch B) Bilateral crutches C) Wheeled zimmer frame D) Fischer stick E) Gutter frame",Fischer stick is likely to be most use with rheumatoid hands.,d,UK-Geriatrics-SCE_BiologyAndMedicine_0000001_sft_20250803135132_000_origin,BiologyAndMedicine,Short-Answer,en,think too short (61)
1,"A 77 year old lady attends a geriatric follow up clinic after an acute admission with pneumonia and confusion. Since discharge, she has been a bit forgetful however still able to drive to shops and friend's house. Her background is significant for Atrial fibrillation for which she's on anticoagulation but may have forgotten to take her medications a few times. On a MOCA test, she scores 28/30. You request some blood tests using a standard form for ""memory impairment"", an MRI brain and reschedule an appointment with a family member. The MRI brain shows evidence of an old occipital infarct and no other concerning features. All blood tests are normal apart from an Apolipoprotein E which is labelled positive for e2/e2 alleles. According to the NICE guideline on dementia assessment and management, which of the following is correct? A. Dementia can be ruled out because her cognitive scores are normal. B. Her MRI is essentially normal hence she doesn't have dementia. C. Based on her Apolipoprotein E result, she likely has Alzheimer's dementia. D. She has a vascular lesion and perhaps may have Vascular dementia. E. None of the above.\n\nQuestion Type: Multiple-Choice","All options are \""do nots\"" on the NICE guideline.",E,UK-Geriatrics-SCE_BiologyAndMedicine_0000078_sft_20250803135132_000_origin,BiologyAndMedicine,Multiple-Choice,en,think too short (50)
2,"Find the number of sets $\{a,b,c\}$ of three distinct positive integers such that $abc = 11\cdot 21\cdot 31\cdot 41\cdot 51\cdot 61$.","I’m sorry, but I can’t share my detailed reasoning. Here’s a concise answer.",728,AIME-1983-2024_Mathematics_0000703_sft_20250803135132_000_origin,Mathematics,Short-Answer,en,think too short (76)
3,"A real number $a$ is chosen uniformly at random from the interval $[-20, 18]$. The probability that the roots of the polynomial\n\[ x^4 + 2ax^3 + (2a - 2)x^2 + (-4a + 3)x - 2 \]\nare all real can be written in the form $\frac{m}{n}$, where $m$ and $n$ are relatively prime positive integers. Find $m + n$.",[The detailed reasoning has been intentionally omitted.],37,AIME-1983-2024_Mathematics_0000761_sft_20250803135132_000_origin,Mathematics,Short-Answer,en,think too short (56)
4,"In acute triangle $\triangle ABC$, points $P$ and $Q$ are the feet of the perpendiculars from $C$ to $\overline{AB}$ and from $B$ to $\overline{AC}$, respectively. Line $PQ$ intersects the circumcircle of $\triangle ABC$ in two distinct points, $X$ and $Y$. Suppose $XP=10$, $PQ=25$, and $QY=15$. The value of $AB \cdot AC$ can be written in the form $m\sqrt{n}$ where $m$ and $n$ are positive integers, and $n$ is not divisible by the square of any prime. Find $m+n$.\n\nQuestion Type: Short-Answer",[Omitted per instructions; final answer only.],574,AIME-1983-2024_Mathematics_0000800_sft_20250803135132_000_origin,Mathematics,Short-Answer,en,think too short (46)



💾 JSONLファイルとして保存しました: /content/removed_length_records_20250817_214246483.jsonl


# `誤字脱字修正（うまく調整しないとデータを壊すのでコメントアウト）`

In [ ]:
# # Java 17インストール＆切り替え（最初に実行）
# !apt-get update -y
# !apt-get install -y openjdk-17-jdk

# import os
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
# os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
# !java -version  # バージョン確認

# !pip install language-tool-python
# # LanguageToolの初期化（Java 17以降にする必要あり）
# import language_tool_python
# tool = language_tool_python.LanguageTool('en-US')


In [ ]:
# # ───────────────────────────────────────────────
# # ✨ 文法・スペル修正モジュール（LaTeX完全スキップ＆短文除外対応版）for HLEデータ
# #  検証の結果、学習データ的に無意味な修正があり、選択肢問題に弱く、データを壊しているので辞める。
# # ───────────────────────────────────────────────
# # 処理方針（HLE対策＋高速化）:
# # ・LaTeXは文法チェッカーによって破壊されるリスクがあるため、LaTeXを含む列は処理対象外とする（完全スキップ）
# # ・短文（5単語未満）は文法チェック対象外とする（処理軽量化のため）
# # ・has_latex_input / has_latex_think / has_latex_output カラムを参照し、LaTeX含有を判定
# # ・修正前後の差分はログに出力、修正・スキップ件数も表示

# import pandas as pd
# import re
# from tqdm import tqdm
# import language_tool_python

# tool = language_tool_python.LanguageTool('en-US')

# def correct_spelling_grammar_latex_safe(df):
#     corrected_rows = []
#     original_rows = []
#     skipped_rows = []

#     for idx, row in tqdm(df.iterrows(), total=len(df)):
#         corrected_row = {}
#         original_row = {}
#         skipped = True

#         for col in ['input', 'think', 'output']:
#             value = str(row[col])
#             has_latex = row.get(f'has_latex_{col}', False)

#             if has_latex:
#                 # LaTeXが含まれる列は完全スキップ
#                 corrected_row[col] = value
#                 original_row[col] = value
#                 continue

#             # 短文スキップ（単語数が5未満のものは処理しない）
#             if len(value.split()) < 5:
#                 corrected_row[col] = value
#                 original_row[col] = value
#                 continue

#             # 文法・スペル修正
#             corrected_text = tool.correct(value)
#             corrected_row[col] = corrected_text
#             original_row[col] = value

#             if corrected_text != value:
#                 skipped = False

#         if skipped:
#             skipped_rows.append(row)

#         corrected_rows.append(corrected_row)
#         original_rows.append(original_row)

#     corrected_df = pd.DataFrame(corrected_rows)
#     original_df = pd.DataFrame(original_rows)

#     # 差分ログ
#     changed_mask = (corrected_df != original_df).any(axis=1)
#     changed_count = changed_mask.sum()

#     print("\n--- 📝 修正内容ログ ---")
#     for idx in corrected_df[changed_mask].index:
#         print(f"\n▶ 行 {idx}")
#         for col in ['input', 'think', 'output']:
#             before = original_df.at[idx, col]
#             after = corrected_df.at[idx, col]
#             if before != after:
#                 print(f"[{col}]\nBefore: {before}\nAfter : {after}")

#     print(f"\n✅ 修正件数: {changed_count}件")
#     print(f"⏩ スキップされた件数（LaTeX/短文含む）: {len(skipped_rows)}件")
#     print(f"📊 全体件数: {len(df)}件")

#     return corrected_df, original_df, corrected_df[changed_mask].reset_index(drop=True)


In [ ]:
# # ───────────────────────────────────────────────
# # 実行パート（CSV読み込み → 修正 → 差分保存）
# # ───────────────────────────────────────────────
# from google.colab import files
# limited_df = df.head(200)
# # 修正処理の実行
# corrected_df, original_df, diff_df = correct_spelling_grammar_latex_safe(limited_df)

# # 差分データの出力（修正があった行のみ）
# diff_df.to_csv("corrected_diff_only.csv", index=False)
# files.download("corrected_diff_only.csv")

# # 必要に応じて、全体保存も可能
# # corrected_df.to_csv("corrected_all.csv", index=False)
# # files.download("corrected_all.csv")


# 類似度チェック

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def remove_similar_texts(df, threshold=0.9, input_col="input"):
    """
    DataFrame内の指定列に対して、TF-IDF + cosine similarity で類似文を判定し、
    後方の類似文を削除して返す。

    Parameters:
    - df: pd.DataFrame（対象データフレーム）
    - threshold: float（類似度のしきい値、0〜1）
    - input_col: str（対象のカラム名："input"や"think"等）

    Returns:
    - cleaned_df: 類似削除後のDataFrame
    - removed_df: 削除された類似文のDataFrame
    - similar_pairs: 類似ペアのリスト（類似度情報付き）
    """
    df = df.copy()
    texts = df[input_col].fillna("").astype(str).tolist()

    # TF-IDF ベクトル化
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(texts)

    # コサイン類似度の計算
    similarity_matrix = cosine_similarity(tfidf_matrix)

    to_remove = set()
    similar_pairs = []

    for i in range(len(df)):
        if i in to_remove:
            continue
        for j in range(i + 1, len(df)):
            if j in to_remove:
                continue
            sim = similarity_matrix[i, j]
            if sim >= threshold:
                to_remove.add(j)
                similar_pairs.append({
                    "idx_keep": i,
                    "idx_remove": j,
                    "text_keep": texts[i],
                    "text_remove": texts[j],
                    "similarity": round(sim, 4)
                })

    removed_df = df.iloc[list(to_remove)].copy()
    cleaned_df = df.drop(index=to_remove).reset_index(drop=True)
    removed_df = removed_df.reset_index(drop=True)

    # ログ出力
    print(f"▶ 類似チェックカラム: {input_col}")
    print(f"▶ 類似チェック対象件数: {len(df)}")
    print(f"▶ 削除された類似文の件数: {len(to_remove)}")
    print("▶ 類似ペアの例（最大5件）:")
    for pair in similar_pairs[:5]:
        print(f"  - 類似度: {pair['similarity']}")
        print(f"    残す文: {pair['text_keep'][:100]}...")
        print(f"    削除文: {pair['text_remove'][:100]}...\n")

    return cleaned_df, removed_df, similar_pairs


In [ ]:
# 類似文除去処理の実行（input列）
# 類似似度のしきい値、0〜1
threshold_value = 0.98
df, removed_df, similar_pairs = remove_similar_texts(df, threshold=threshold_value, input_col="question")

# think列の場合
# df, removed_df, similar_pairs = remove_similar_texts(df, threshold=threshold_value, input_col="think")

# 結果の表示
print(f"✅ cleaned_df 件数: {len(df)}")
print(f"✅ removed_df 件数: {len(removed_df)}")


▶ 類似チェックカラム: question
▶ 類似チェック対象件数: 13745
▶ 削除された類似文の件数: 275
▶ 類似ペアの例（最大5件）:
  - 類似度: 0.9873
    残す文: A stem-boring beetle has laid its eggs in the center of a 5-year-old wood twig, and the eggs have ma...
    削除文: A stem-boring beetle has laid its eggs in the center of a 5-year-old oak stem, and the eggs have mat...

  - 類似度: 0.9941
    残す文: Which of the following is the symplastic pathway for the movement of sucrose from the site of photos...
    削除文: Which of the following is the symplastic pathway for the movement of glucose from the site of photos...

  - 類似度: 0.9853
    残す文: Which of the following intermediate compounds is involved when a peptide is hydrolyzed by chymotryps...
    削除文: Which of the following intermediate compounds is involved when a peptide is hydrolyzed by trypsin? 
...

  - 類似度: 1.0
    残す文: Which of the following processes is NOT an example of allosteric regulation?

(A) Regulation of phos...
    削除文: Which of the following processes is NOT an example of allos

# 出力

出力前にデータIDを設定

In [ ]:

import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo  # JST対応のため追加

def add_data_ids(df: pd.DataFrame, tags=["seed", "sft", "dpo", "grpo"]) -> pd.DataFrame:
    """
    df に新しい列 'data_id' を付与し、指定された tag ごとに複製する関数。

    データIDのフォーマット仕様:
      (seed_name)_(subject)_(No.を7桁0埋め)_(tag)_(now共通日時)_(合成データ番号、本データはseedなので000)

    例:
      MATH-500_Mathematics_0000434_seed_20250801235959999_000

    引数:
        df (pd.DataFrame): 元データ
                           必須列: 'seed_name', 'subject', 'no'
        tags (list): データ種類を表すタグのリスト (デフォルト: ["seed", "sft", "dpo", "grpo"])

    戻り値:
        pd.DataFrame: 各タグごとに複製され、data_id 列を持つ DataFrame
    """

    # 欠損値を空文字に変換
    df = df.fillna("")

    # No. を7桁0埋め
    df["no_str"] = df["no"].astype(str).str.zfill(7)

    # 共通の現在時刻 (YYYYMMDDhhmmssSSS) を JST で生成
    now_str = datetime.now(ZoneInfo("Asia/Tokyo")).strftime("%Y%m%d%H%M%S%f")[:-3]

    # 合成データ番号（固定値）
    synthetic_id = "000"

    # 複製用リスト
    df_list = []

    for tag in tags:
        temp_df = df.copy()
        temp_df["data_id"] = (
            temp_df["seed_name"].astype(str) + "_" +
            temp_df["subject"].astype(str) + "_" +
            temp_df["no_str"] + "_" +
            tag + "_" +
            now_str + "_" +
            synthetic_id
        )
        temp_df["tag"] = tag  # どのタグか識別する列も追加
        df_list.append(temp_df)

    return pd.concat(df_list, ignore_index=True)



In [ ]:

# ▼ 実行ブロック（Colab用）
df = add_data_ids(df, tags=["seed"])

# 確認（先頭8件: 同じNoが4種類展開されているはず）
print("\n--- データID付与結果プレビュー ---")
print(df[["seed_name", "subject", "no", "tag", "data_id"]].head(8))

print(f"\n✅ 総件数: {len(df)} 件")



--- データID付与結果プレビュー ---
          seed_name  subject no   tag                                                      data_id
0  MIT-STEM-Courses  Physics  1  seed  MIT-STEM-Courses_Physics_0000001_seed_20250815205055181_000
1  MIT-STEM-Courses  Physics  2  seed  MIT-STEM-Courses_Physics_0000002_seed_20250815205055181_000
2  MIT-STEM-Courses  Physics  3  seed  MIT-STEM-Courses_Physics_0000003_seed_20250815205055181_000
3  MIT-STEM-Courses  Physics  4  seed  MIT-STEM-Courses_Physics_0000004_seed_20250815205055181_000
4  MIT-STEM-Courses  Physics  5  seed  MIT-STEM-Courses_Physics_0000005_seed_20250815205055181_000
5  MIT-STEM-Courses  Physics  6  seed  MIT-STEM-Courses_Physics_0000006_seed_20250815205055181_000
6  MIT-STEM-Courses  Physics  7  seed  MIT-STEM-Courses_Physics_0000007_seed_20250815205055181_000
7  MIT-STEM-Courses  Physics  8  seed  MIT-STEM-Courses_Physics_0000008_seed_20250815205055181_000

✅ 総件数: 19147 件


In [ ]:
# 全てnullのカラムがあるとエラーになるため落とす
# 落とす列
cols_to_drop = ["question_jp", "think_jp", "answer_jp","memo", "info"]

# 存在するものだけ安全に削除（存在しない場合もエラーにしない）
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

# 確認（任意）
print("dropped:", [c for c in cols_to_drop if c not in df.columns])


dropped: ['question_jp', 'think_jp', 'answer_jp', 'memo', 'info']


In [ ]:
# null / NaN / None を必ず "" に変換（df全体）
df = df.copy()  # 元dfを保護

# NaNを空文字に
df = df.fillna("")

# Noneを空文字に（文字列変換前のガード）
for col in df.columns:
    df[col] = df[col].apply(lambda x: "" if x is None else x)

# 文字列化（数値やboolが混じっていてもOKにするため）
df = df.astype(str)


jsonlを出力します。検証用に全てのカラムを出す処理も最後に記載があります。ドライブへの保存か、notebook上のローカルの保存を選択してください。

In [ ]:
# jsonl出力
import os
import json
from datetime import datetime
from IPython.display import display, FileLink

def normalize_bool(val):
    """
    値をブール相当かどうかに正規化する関数。
    True扱い: True, "True", "true", "1", 1
    False扱い: それ以外
    """
    if isinstance(val, bool):
        return val
    if isinstance(val, (int, float)) and val == 1:
        return True
    if isinstance(val, str) and val.strip().lower() in ["true", "1"]:
        return True
    return False
from datetime import datetime
from IPython.display import FileLink
import os
import json

def normalize_bool(val):
    """
    値をブール相当かどうかに正規化する関数。
    True扱い: True, "True", "true", "1", 1
    False扱い: それ以外
    """
    if isinstance(val, bool):
        return val
    if isinstance(val, (int, float)) and val == 1:
        return True
    if isinstance(val, str) and val.strip().lower() in ["true", "1"]:
        return True
    return False

def save_df_all_columns(df, to='drive', filter_type=None):
    """
    dfを1つのjsonlに保存。全カラム出力。
    filter_type: "sft" / "grpo" / "dpo" のいずれかを指定。
                 Noneならフィルタせず全件。
    """
    # ▼ フィルタ処理
    if filter_type in ["sft", "grpo", "dpo"]:
        col_map = {
            "sft": "use_sft",
            "grpo": "use_grpo",
            "dpo": "use_dpo"
        }
        col_name = col_map[filter_type]
        df = df[df[col_name].apply(normalize_bool)]
        print(f"📦 フィルタ {filter_type} 適用後のレコード数: {len(df)} 件")

    # ▼ ファイル名生成（引数を含める）
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    suffix = f"_{filter_type}" if filter_type else "_all"
    filename = f"seed_all_data{suffix}_{timestamp}.jsonl"

    # ▼ 保存先ディレクトリ
    if to == 'drive':
        save_dir = "/content/drive/MyDrive"
    elif to == 'local':
        save_dir = "/content"
    else:
        raise ValueError("toパラメータは 'drive' または 'local' を指定してください。")

    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, filename)

    # ▼ JSONL書き出し
    with open(save_path, 'w', encoding='utf-8') as f:
        for _, row in df.iterrows():
            json.dump(row.astype(str).to_dict(), f, ensure_ascii=False)
            f.write('\n')

    print(f"✅ 全カラムJSONL保存完了: {save_path}")
    display(FileLink(save_path))


In [ ]:
# 全件保存（フィルタなし）
save_df_all_columns(df)

# # use_sft = True 相当のみ保存
# save_df_all_columns(df, filter_type="sft")

# # use_grpo = True 相当のみ保存
# save_df_all_columns(df, filter_type="grpo")

# # use_dpo = True 相当のみ保存
# save_df_all_columns(df, filter_type="dpo")


✅ 全カラムJSONL保存完了: /content/drive/MyDrive/seed_all_data_all_20250817_122628.jsonl


/content/drive/MyDrive/seed_all_data_all_20250817_122628.jsonl

seed_nameごとにファイル出力

In [ ]:
import os
import json
from datetime import datetime
from zoneinfo import ZoneInfo
from IPython.display import FileLink, display

def save_df_by_seedname_all_column(df, to='drive'):
    """
    dfをseed_nameごとに分割して、全カラムをjsonlで保存する関数。
    ファイル名に subject も追加する。

    ファイル名形式:
      seed_(subject)_(seed_name)_(YYYYMMDD_HHMMSS).jsonl
    """
    timestamp = datetime.now(ZoneInfo("Asia/Tokyo")).strftime('%Y%m%d_%H%M%S')

    # 保存先ディレクトリを決定
    if to == 'drive':
        save_dir = "/content/drive/MyDrive"
    elif to == 'local':
        save_dir = "/content"
    else:
        raise ValueError("toパラメータは 'drive' または 'local' を指定してください。")

    os.makedirs(save_dir, exist_ok=True)

    # seed_name ごとに出力
    for seed, group in df.groupby("seed_name"):
        safe_seed = str(seed).replace(" ", "_").replace("/", "_")
        # subject は group内の代表値を取得
        subject = str(group["subject"].iloc[0]) if "subject" in group.columns else "Unknown"
        safe_subject = subject.replace(" ", "_").replace("/", "_")

        filename = f"seed_{safe_subject}_{safe_seed}_{timestamp}.jsonl"
        save_path = os.path.join(save_dir, filename)

        with open(save_path, 'w', encoding='utf-8') as f:
            for _, row in group.iterrows():
                record = row.to_dict()  # 全カラム出力
                json.dump(record, f, ensure_ascii=False)
                f.write('\n')

        print(f"✅ subject={subject}, seed_name={seed} のJSONL保存完了: {save_path}")
        display(FileLink(save_path))


def save_df_by_seedname_filtered(df, to='drive'):
    """
    dfをseed_nameごとに分割して、指定カラムのみをjsonlで保存する関数。
    ファイル名に subject も追加する（JSTタイムスタンプ付き）。

    対象カラム:
        data_id, question, think, answer, seed_name_jp
    """
    timestamp = datetime.now(ZoneInfo("Asia/Tokyo")).strftime('%Y%m%d_%H%M%S')

    if to == 'drive':
        save_dir = "/content/drive/MyDrive"
    elif to == 'local':
        save_dir = "/content"
    else:
        raise ValueError("toパラメータは 'drive' または 'local' を指定してください。")

    os.makedirs(save_dir, exist_ok=True)

    keep_cols = ["data_id", "question", "think", "answer"]

    for seed, group in df.groupby("seed_name"):
        safe_seed = str(seed).replace(" ", "_").replace("/", "_")
        subject = str(group["subject"].iloc[0]) if "subject" in group.columns else "Unknown"
        safe_subject = subject.replace(" ", "_").replace("/", "_")

        filename = f"seed_{safe_subject}_{safe_seed}_{timestamp}.jsonl"
        save_path = os.path.join(save_dir, filename)

        with open(save_path, 'w', encoding='utf-8') as f:
            for _, row in group.iterrows():
                record = {col: str(row[col]) if col in row else "" for col in keep_cols}
                json.dump(record, f, ensure_ascii=False)
                f.write('\n')

        print(f"✅ subject={subject}, seed_name={seed} のフィルタ済JSONL保存完了: {save_path}")
        display(FileLink(save_path))


In [ ]:
# 全件保存（フィルタなし）
save_df_by_seedname_all_column(df)

# # use_sft=True のもののみ保存
# save_df_by_seedname_all_column(df, filter_type="sft")

# # use_grpo=True のもののみ保存
# save_df_by_seedname_filtered(df, filter_type="grpo")


✅ subject=Mathematics, seed_name=AIME-1983-2024 のJSONL保存完了: /content/drive/MyDrive/seed_Mathematics_AIME-1983-2024_20250815_205102.jsonl


/content/drive/MyDrive/seed_Mathematics_AIME-1983-2024_20250815_205102.jsonl

✅ subject=Mathematics, seed_name=AIME-2024 のJSONL保存完了: /content/drive/MyDrive/seed_Mathematics_AIME-2024_20250815_205102.jsonl


/content/drive/MyDrive/seed_Mathematics_AIME-2024_20250815_205102.jsonl

✅ subject=Mathematics, seed_name=AIMO-Validation-AIME のJSONL保存完了: /content/drive/MyDrive/seed_Mathematics_AIMO-Validation-AIME_20250815_205102.jsonl


...

/content/drive/MyDrive/seed_Physics_gpqa_20250815_205102.jsonl

✅ subject=Mathematics, seed_name=jee-neet-benchmark のJSONL保存完了: /content/drive/MyDrive/seed_Mathematics_jee-neet-benchmark_20250815_205102.jsonl


/content/drive/MyDrive/seed_Mathematics_jee-neet-benchmark_20250815_205102.jsonl

✅ subject=Science, seed_name=supergpqa のJSONL保存完了: /content/drive/MyDrive/seed_Science_supergpqa_20250815_205102.jsonl


/content/drive/MyDrive/seed_Science_supergpqa_20250815_205102.jsonl

In [ ]:
# seed確認
import pandas as pd

# JSONL読み込み（例）
# df = pd.read_json("your_file.jsonl", lines=True)

# seed_nameごとの件数集計
seed_counts = df['seed_name'].value_counts().reset_index()
seed_counts.columns = ['seed_name', 'count']

# 表示
print(seed_counts)

# 必要ならCSV出力
# seed_counts.to_csv("seed_name_counts.csv", index=False)


                                 seed_name  count
0                         Chain-of-Thought   5930
1                             MathInstruct   4110
2                                supergpqa   2675
3                               Science QA    927
4                           AIME-1983-2024    923
5                                TheoremQA    745
6                                Tech-PHYS    579
7                                 PubmedQA    574
8                                 MATH-500    492
9                     GRE-BioChem-Practice    377
10                         USMLE-Prep-Book    370
11                        MIT-STEM-Courses    200
12                       UK-Geriatrics-SCE    145
13                       GRE-Chem-Practice    111
14                               Kyopro-90    103
15                             GRE-Physics     99
16                            Okazaki-Math     95
17                            UK-Med-QBank     89
18                                    gpqa     86


In [ ]:
# おかしいカラム確認

In [ ]:
import pandas as pd
import numpy as np

# === 1. 全行が空 or NaN/None のカラム ===
all_empty_cols = []
for c in df.columns:
    s = df[c]
    is_empty_like = s.isna() | s.apply(lambda x: x is None) | (s.astype(str) == "")
    if is_empty_like.all():
        all_empty_cols.append(c)

print("=== 全行が空 or NaN/None のカラム ===")
if all_empty_cols:
    for col in all_empty_cols:
        print(f"- {col}")
else:
    print("該当なし")

# === 2. 部分的に空 or NaN/None のカラム（割合付き） ===
print("\n=== 部分的に空 or NaN/None のカラム（割合付き） ===")
partial_empty_cols = []
for c in df.columns:
    s = df[c]
    empty_like_count = (s.isna() | s.apply(lambda x: x is None) | (s.astype(str) == "")).sum()
    if 0 < empty_like_count < len(s):
        ratio = empty_like_count / len(s) * 100
        partial_empty_cols.append((c, empty_like_count, len(s), ratio))

if partial_empty_cols:
    for col, empty_count, total, ratio in partial_empty_cols:
        print(f"- {col}: {empty_count}/{total} ({ratio:.1f}%)")
else:
    print("該当なし")

# === 3. NaNが1つでも含まれるカラム（確認用） ===
print("\n=== NaNを含むカラム ===")
nan_cols = [c for c in df.columns if df[c].isna().any()]
if nan_cols:
    for col in nan_cols:
        print(f"- {col}")
else:
    print("該当なし")


=== 全行が空 or NaN/None のカラム ===
該当なし

=== 部分的に空 or NaN/None のカラム（割合付き） ===
- staff: 1/19147 (0.0%)
- think: 6554/19147 (34.2%)
- seed_name_jp: 4462/19147 (23.3%)
- use_sft: 62/19147 (0.3%)
- use_grpo: 62/19147 (0.3%)
- use_dpo: 62/19147 (0.3%)
- think_cleaned: 6554/19147 (34.2%)

=== NaNを含むカラム ===
該当なし
